# Day 5 模块 3：模型证据（重要性 + what-if）

模型在汇报里只干两件事：

1. **优先盯什么**（特征重要性 → 业务话）
2. **拧旋钮会怎样**（what-if；下午网页上再点一遍）

重要性 ≠ 严格因果；what-if ≠ 明天准账。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day5_cafe_sales.csv'),
    Path('day5') / 'day5_cafe_sales.csv',
    Path('教学课程') / 'day5' / 'day5_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day5_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)
df['weekend_label'] = df['is_weekend'].map({0: '工作日', 1: '周末'})
df['promo_label'] = df['promotion'].map({0: '无促销', 1: '有促销'})
print(df[['day', 'weather_label', 'weekend_label', 'promo_label', 'sales']].head())
print('平均营收:', round(df['sales'].mean(), 1))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

feature_cols = [
    'price', 'promotion', 'is_weekend', 'temp', 'quality',
    'competitors', 'holiday', 'location', 'weather_score',
]
X = df[feature_cols]
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf = RandomForestRegressor(
    n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print('测试 R² :', round(r2_score(y_test, pred), 3))
print('测试 MAE:', round(mean_absolute_error(y_test, pred), 1))


## 1. 重要性 → 跟老板怎么说


In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.round(3))
print('Top3:', list(imp.head(3).index))


### 话术模板

```text
在这批历史上，模型做预测时更依赖：A、B、C。
所以日常复盘我建议优先盯这几项。
但「依赖高」不等于「改它就一定涨」——还要结合现场。
```

R² / MAE 白话：

```text
测试集上大约能解释七成多波动；
单日预测常偏几百块很正常，用来比较方案、排优先级。
```


## 2. What-if（网页演示的预演）


In [ ]:
base = {
    'price': 25.0, 'promotion': 0, 'is_weekend': 0, 'temp': 22.0,
    'quality': 7.5, 'competitors': 2, 'holiday': 0, 'location': 7.0,
    'weather_score': 1.0,
}

def predict_one(overrides=None):
    row = base.copy()
    if overrides:
        row.update(overrides)
    x = pd.DataFrame([row], columns=feature_cols)
    return round(float(rf.predict(x)[0]), 1)

print('基准', predict_one())
print('有促销', predict_one({'promotion': 1}))
print('周末', predict_one({'is_weekend': 1}))
print('竞品+2', predict_one({'competitors': 4}))
print('大雨', predict_one({'weather_score': 0.3}))
print('单价+5', predict_one({'price': 30}))


## 3. 小练习

1. Top3 各半句业务话。
2. 若只能给老板现场拧**一个**旋钮，你拧哪个？准备说什么？
3. 记下一组「基准 vs 改后」数字，模块 5 汇报要用。


In [ ]:
# 在这里写
